# 05. Validate the complete synthetic build

This final job gate verifies regenerated data, Lakeflow outputs, governance controls, the ML model and endpoint, GenAI briefs, generator economics, and an optional Genie Agent. It fails the workflow on missing proof so a successful job run is evidence that the synthetic demo is ready.


In [ ]:
import json
import time

import mlflow
from databricks.sdk import WorkspaceClient


dbutils.widgets.text("catalog", "main", "Target catalog")
dbutils.widgets.text("schema", "gpo_spend_intelligence", "Target schema")
dbutils.widgets.text("endpoint", "gpo-maverick-risk", "Serving endpoint")
dbutils.widgets.text("genie_space_id", "", "Genie Agent ID")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
ENDPOINT = dbutils.widgets.get("endpoint")
GENIE_SPACE_ID = dbutils.widgets.get("genie_space_id")
FQ = f"{CATALOG}.{SCHEMA}"
RESULTS = []


def scalar(statement):
    row = spark.sql(statement).first()
    return row[0] if row else None


def check(name, condition, observed):
    ok = bool(condition)
    RESULTS.append({"name": name, "passed": ok, "observed": observed})
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: {observed}")


spend_count = int(scalar(f"SELECT count(*) FROM {FQ}.spend_lines"))
check("spend_lines row count", spend_count == 60000, spend_count)

dim_counts = {
    table: int(scalar(f"SELECT count(*) FROM {FQ}.{table}"))
    for table in ("dim_members", "dim_suppliers", "dim_contracts", "ref_categories")
}
check("dimensions populated", all(value > 0 for value in dim_counts.values()), dim_counts)

bronze = int(scalar(f"SELECT count(*) FROM {FQ}.po_events_bronze_sdp"))
silver = int(scalar(f"SELECT count(*) FROM {FQ}.po_events_silver_sdp"))
check("Lakeflow bronze equals silver", bronze == silver == 500, {"bronze": bronze, "silver": silver})

off_contract_rate = float(scalar(
    f"SELECT avg(CASE WHEN on_contract THEN 0 ELSE 1 END) FROM {FQ}.spend_lines"
))
check("off-contract rate plausible", 0.10 <= off_contract_rate <= 0.35, round(off_contract_rate, 4))

model_name = f"{FQ}.maverick_risk_model"
model_version = mlflow.MlflowClient().get_model_version_by_alias(model_name, "champion")
check("UC model champion alias", bool(model_version.version), f"v{model_version.version}")

scored = int(scalar(f"SELECT count(*) FROM {FQ}.gold_maverick_risk"))
tiers = {row[0] for row in spark.sql(f"SELECT DISTINCT risk_tier FROM {FQ}.gold_maverick_risk").collect()}
check("all spend lines scored", scored == spend_count and tiers == {"Low", "Medium", "High"}, {"rows": scored, "tiers": sorted(tiers)})

masks = spark.sql(
    f"SELECT column_name, mask_name FROM {CATALOG}.information_schema.column_masks "
    f"WHERE table_schema='{SCHEMA}' AND table_name='spend_lines'"
).collect()
check("member-name mask registered", len(masks) == 1, [row.asDict() for row in masks])

masked_name = scalar(f"SELECT member_name FROM {FQ}.spend_lines LIMIT 1")
check("member-name mask enforced", masked_name == "***REDACTED***", masked_name)

row_filters = spark.sql(
    f"SELECT table_name, filter_name FROM {CATALOG}.information_schema.row_filters "
    f"WHERE table_schema='{SCHEMA}' AND table_name='gold_review_queue'"
).collect()
check("regional row filter registered", len(row_filters) == 1, [row.asDict() for row in row_filters])

visible_regions = [row[0] for row in spark.sql(
    f"SELECT DISTINCT region FROM {FQ}.gold_review_queue ORDER BY region"
).collect()]
check("row-filtered review queue visible", len(visible_regions) >= 1, visible_regions)

briefs = int(scalar(f"SELECT count(*) FROM {FQ}.gold_action_briefs"))
avg_quality = float(scalar(f"SELECT avg(actionability_score) FROM {FQ}.gold_action_brief_quality"))
check("GenAI action briefs generated", briefs == 25, briefs)
check("LLM-as-judge quality gate", avg_quality >= 4.0, round(avg_quality, 2))

incremental_fee = float(scalar(f"SELECT incremental_gpo_fee_m FROM {FQ}.gold_gpo_economics"))
check("coverage-to-revenue economics", incremental_fee > 0, f"${incremental_fee:.2f}M")

w = WorkspaceClient()
endpoint_state = w.serving_endpoints.get(ENDPOINT).state.ready.value
record = {
    "member_id": "MBR-18",
    "region": "Midwest",
    "supplier_id": "SUP-050",
    "category": "Med-Surg Supplies",
    "quantity": 12,
    "unit_price": 900.0,
    "amount": 10800.0,
    "days_to_expiry": 20,
    "order_month": 11,
}
prediction = None
for attempt in range(4):
    try:
        response = w.serving_endpoints.query(name=ENDPOINT, dataframe_records=[record])
        prediction = response.predictions
        break
    except Exception:
        if attempt == 3:
            raise
        time.sleep(10)
check("serving endpoint live inference", endpoint_state == "READY" and bool(prediction), {"state": endpoint_state, "prediction": prediction})

message = w.genie.start_conversation_and_wait(
    GENIE_SPACE_ID,
    "Which member has the highest off-contract spend, and how much is it?",
)
generated_sql = ""
answer = ""
for attachment in message.attachments or []:
    if getattr(attachment, "query", None):
        generated_sql = attachment.query.query or ""
    if getattr(attachment, "text", None):
        answer = attachment.text.content or ""
correct_answer = "10.7" in answer and ("MBR-18" in answer or "REDACTED" in answer)
check("Genie generated SQL and governed answer", bool(generated_sql) and correct_answer, {"answer": answer, "sql": generated_sql})

failures = [result for result in RESULTS if not result["passed"]]
summary = {
    "catalog": CATALOG,
    "schema": SCHEMA,
    "passed": len(RESULTS) - len(failures),
    "total": len(RESULTS),
    "failures": failures,
    "results": RESULTS,
}
print(json.dumps(summary, indent=2, default=str))
if failures:
    raise AssertionError(f"{len(failures)} validation checks failed")

dbutils.notebook.exit(json.dumps(summary, default=str))